# LAMOST star heatmap detector

Train ResNet-50 + FPN + heatmap head on a CUDA-enabled Colab runtime. Before running, select **Runtime → Change runtime type → T4 GPU**.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before continuing.'
print(torch.cuda.get_device_name(0))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Upload the exported `LAMOST_subset` directory to `MyDrive/datasets/LAMOST_subset`. It must contain `train/images`, `dev/images`, `test/images`, and `manifests`.

In [ ]:
from pathlib import Path
import shutil
from tqdm.auto import tqdm

DRIVE_SEARCH_ROOT = Path('/content/drive/MyDrive/datasets/LAMOST_subset')
DATASET_ROOT = Path('/content/LAMOST_subset')
OUTPUT_DIR = Path('/content/drive/MyDrive/starmapFusion_outputs/resnet50_fpn_heatmap')

train_manifests = list(DRIVE_SEARCH_ROOT.rglob('train.jsonl'))
if not train_manifests:
    raise FileNotFoundError(f'No train.jsonl found under {DRIVE_SEARCH_ROOT}')
if len(train_manifests) > 1:
    print('Multiple manifests found; using:', train_manifests[0])
DRIVE_MANIFEST_DIR = train_manifests[0].parent
DRIVE_DATASET_ROOT = DRIVE_MANIFEST_DIR.parent

source_files = [path for path in DRIVE_DATASET_ROOT.rglob('*') if path.is_file()]
copied = 0
skipped = 0
for source_path in tqdm(source_files, desc='Syncing dataset', unit='file'):
    relative_path = source_path.relative_to(DRIVE_DATASET_ROOT)
    destination_path = DATASET_ROOT / relative_path
    destination_path.parent.mkdir(parents=True, exist_ok=True)
    if destination_path.is_file() and destination_path.stat().st_size == source_path.stat().st_size:
        skipped += 1
        continue
    shutil.copy2(source_path, destination_path)
    copied += 1

MANIFEST_DIR = DATASET_ROOT / 'manifests'
required = ['train.jsonl', 'validation.jsonl', 'test.jsonl']
missing = [name for name in required if not (MANIFEST_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f'Missing local manifests: {missing}')

print(f'Copied: {copied}, already present: {skipped}')
print('Detected Drive root:', DRIVE_DATASET_ROOT)
print('Local dataset:', DATASET_ROOT)
print('Manifests:', MANIFEST_DIR)
print('Outputs:', OUTPUT_DIR)


In [ ]:
!git clone --branch feature/star-detection https://github.com/antonovchynnikov13/starmapFusion.git /content/starmapFusion
%cd /content/starmapFusion
!pip install -q -r requirements.txt


In [ ]:
!python3 scripts/train_star_detector.py \
  --dataset-root "{DATASET_ROOT}" \
  --manifest-dir "{MANIFEST_DIR}" \
  --output-dir "{OUTPUT_DIR}" \
  --epochs 30 \
  --batch-size 2 \
  --workers 2 \
  --image-size 1024


In [ ]:
import json
history = json.loads((OUTPUT_DIR / 'history.json').read_text())
history[-1], OUTPUT_DIR / 'best.pt'
